In [ ]:
!pip install transformers accelerate pillow pandas numpy scikit-learn lightgbm joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import torch
import time
import re
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from transformers import AutoModel, AutoProcessor
from PIL import Image
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Colab_Notebooks/data.csv")
print("Размер:", df.shape)
print("Пропуски:\n", df.isnull().sum())
print("Распределение label:\n", df['label'].value_counts(normalize=True))
print("Распределение category:\n", df['category'].value_counts())

df['description'] = df['description'].fillna('')
df['text'] = df['name'] + ' ' + df['description'] + ' ' + df['category']

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
print(f"Train size: {len(train_df)}, Val size: {len(val_df)}")

Размер: (12971, 6)
Пропуски:
 Unnamed: 0      0
id              0
name            0
description    19
category        0
label           0
dtype: int64
Распределение label:
 label
0    0.555778
1    0.444222
Name: proportion, dtype: float64
Распределение category:
 category
БАД                      7469
Легковоспламеняющиеся    5502
Name: count, dtype: int64
Train size: 10376, Val size: 2595


In [ ]:
model_embed = AutoModel.from_pretrained(
    "Qwen/Qwen3-VL-Embedding-2B",
    torch_dtype=torch.float16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-Embedding-2B")
print("Модель загружена на:", model_embed.device)

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 4.26GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

Модель загружена на: cuda:0


In [ ]:
def get_text_embedding(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        text = "empty description"
    inputs = processor(text=text, return_tensors="pt", padding=True,
                       truncation=True, max_length=512).to(model_embed.device)
    with torch.no_grad():
        outputs = model_embed(**inputs)
    return outputs.last_hidden_state.mean(dim=1).cpu().numpy().flatten()

In [ ]:
#Тренировочные
CHECKPOINT_DIR = "/content/drive/MyDrive/Colab_Notebooks/embeddings/"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

embeddings_path = CHECKPOINT_DIR + "X_train_emb.npy"
ids_path = CHECKPOINT_DIR + "processed_ids.npy"

if os.path.exists(embeddings_path) and os.path.exists(ids_path):
    X_train_emb = np.load(embeddings_path).tolist()
    processed_ids = set(np.load(ids_path).tolist())
    print(f"Найдены сохранённые данные: {len(processed_ids)} записей.")
else:
    X_train_emb = []
    processed_ids = set()
    print("Начинаем с нуля.")

total = len(train_df)
batch_size = 100
for start_idx in range(0, total, batch_size):
    end_idx = min(start_idx + batch_size, total)
    for idx, row in train_df.iloc[start_idx:end_idx].iterrows():
        if idx in processed_ids:
            continue
        try:
            emb = get_text_embedding(row['text'])
            X_train_emb.append(emb)
            processed_ids.add(idx)
        except Exception as e:
            print(f"Ошибка {idx}: {e}")
    np.save(embeddings_path, np.array(X_train_emb))
    np.save(ids_path, np.array(list(processed_ids)))
    print(f"Обработано {len(processed_ids)} / {total}")

X_train_emb = np.array(X_train_emb)
y_train = train_df['label'].values
np.save("/content/drive/MyDrive/Colab_Notebooks/X_train_text_final.npy", X_train_emb)
np.save("/content/drive/MyDrive/Colab_Notebooks/y_train.npy", y_train)
print("Текстовые эмбеддинги трейна готовы.")

Найдены сохранённые данные: 10376 записей.
Обработано 10376 / 10376
Обработано 10376 / 10376
Обработано 10376 / 10376
Обработано 10376 / 10376
Обработано 10376 / 10376
Обработано 10376 / 10376
Обработано 10376 / 10376
Обработано 10376 / 10376
Обработано 10376 / 10376
Обработано 10376 / 10376
Обработано 10376 / 10376


KeyboardInterrupt: 

In [ ]:
#Валидационные
CHECKPOINT_DIR_VAL = "/content/drive/MyDrive/Colab_Notebooks/embeddings_val/"
os.makedirs(CHECKPOINT_DIR_VAL, exist_ok=True)
embeddings_path_val = CHECKPOINT_DIR_VAL + "X_val_emb.npy"
ids_path_val = CHECKPOINT_DIR_VAL + "processed_ids_val.npy"

if os.path.exists(embeddings_path_val) and os.path.exists(ids_path_val):
    X_val_emb = np.load(embeddings_path_val).tolist()
    processed_ids_val = set(np.load(ids_path_val).tolist())
else:
    X_val_emb = []
    processed_ids_val = set()

total_val = len(val_df)
for start_idx in range(0, total_val, 100):
    end_idx = min(start_idx+100, total_val)
    for idx, row in val_df.iloc[start_idx:end_idx].iterrows():
        if idx in processed_ids_val:
            continue
        try:
            emb = get_text_embedding(row['text'])
            X_val_emb.append(emb)
            processed_ids_val.add(idx)
        except Exception as e:
            print(f"Ошибка {idx}: {e}")
    np.save(embeddings_path_val, np.array(X_val_emb))
    np.save(ids_path_val, np.array(list(processed_ids_val)))
    print(f"Обработано {len(processed_ids_val)} / {total_val}")

X_val_emb = np.array(X_val_emb)
y_val = val_df['label'].values
np.save("/content/drive/MyDrive/Colab_Notebooks/X_val_text_final.npy", X_val_emb)
np.save("/content/drive/MyDrive/Colab_Notebooks/y_val.npy", y_val)
print("Текстовые эмбеддинги валидации готовы.")

Обработано 2595 / 2595
Обработано 2595 / 2595
Обработано 2595 / 2595
Обработано 2595 / 2595
Обработано 2595 / 2595
Обработано 2595 / 2595
Обработано 2595 / 2595


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.


KeyboardInterrupt



In [ ]:
import lightgbm as lgb
from sklearn.metrics import f1_score, classification_report

X_train = np.load("/content/drive/MyDrive/Colab_Notebooks/X_train_text_final.npy")
y_train = np.load("/content/drive/MyDrive/Colab_Notebooks/y_train.npy")
X_val   = np.load("/content/drive/MyDrive/Colab_Notebooks/X_val_text_final.npy")
y_val   = np.load("/content/drive/MyDrive/Colab_Notebooks/y_val.npy")

assert len(X_train) == len(train_df), f"train: {len(X_train)} != {len(train_df)}"
assert len(X_val)   == len(val_df),   f"val: {len(X_val)} != {len(val_df)}"

train_texts = (train_df['name'].fillna('') + ' ' + train_df['description'].fillna('')).str.lower().values
val_texts   = (val_df['name'].fillna('') + ' ' + val_df['description'].fillna('')).str.lower().values

PATTERNS = [
    r'\bбад\b|\bбады\b|\bбадов\b|\bбадами\b',
    r'биологически активн\w* добавк\w*',
    r'dietary supplement',
    r'пищев\w* добавк\w*',
    r'пробиотик|пробиотическ|биотин|омега|коллаген|рыбий жир',
    r'витамин|минерал|экстракт|пищев\w* волокон',
    r'аминокислот|bcaa|протеин|сывороточн|карнитин|креатин|гейнер|предтрен|спортивн\w* питан',
    r'не является бад|не биодобавк|не является биологически',
    r'спичк|зажигалк|огнив|керосин',
    r'горюче|топлив|бензин|газ|пропан|бутан|ацетон|спирт|розжиг',
    r'аэрозол|спрей|баллон',
    r'мангал|гриль|барбекю|плита|духовк|камин|печь|горелк',
    r'уголь\w*.{0,30}(рисован|фильтр|кальян)|активированн\w* уголь',
    r'без содержим|без газа|без топлива|без баллона|пустой',
]

def rule_features(texts):
    F = np.zeros((len(texts), len(PATTERNS)), dtype=np.float32)
    for i, t in enumerate(texts):
        for j, p in enumerate(PATTERNS):
            F[i, j] = int(bool(re.search(p, t)))

    mark  = np.clip(F[:,0] + F[:,1] + F[:,2], 0, 1)  # маркировка БАД
    sport = F[:,6]
    neg   = F[:,7]
    supp  = np.clip(F[:,4] + F[:,5], 0, 1)           # витаминный лексикон
    food  = F[:,3]

    G = np.column_stack([
        F,
        mark * sport,               # спортпит С маркировкой -> бан (приоритет)
        mark * (1-sport) * (1-neg), # чистая маркировка -> не бан
        (1-mark) * supp,            # добавки БЕЗ маркировки -> бан
        (1-mark) * food,            # "пищевая добавка" БЕЗ маркировки -> бан
        sport,
        neg,
    ]).astype(np.float32)
    return G

X_train_f = np.hstack([X_train, rule_features(train_texts)])
X_val_f   = np.hstack([X_val,   rule_features(val_texts)])
print("Размерность признаков:", X_train_f.shape[1])

train_bad  = (train_df['category'] == 'БАД').values
val_bad    = (val_df['category'] == 'БАД').values
train_flam = ~train_bad
val_flam   = ~val_bad

Fb = rule_features(train_texts[train_bad])[:, :len(PATTERNS)]
yb = y_train[train_bad]
mark  = np.clip(Fb[:,0]+Fb[:,1]+Fb[:,2],0,1)
sport = Fb[:,6]; neg = Fb[:,7]
supp  = np.clip(Fb[:,4]+Fb[:,5],0,1)

def stat(cond, name):
    n = int(cond.sum())
    if n: print(f"{name:34s} n={n:5d}  доля label=1: {yb[cond].mean():.3f}")

print("--- Чистота правил (БАД, train) ---")
stat(sport==1,                      "спортпит")
stat((mark==1)&(sport==0)&(neg==0), "чистая маркировка")
stat((mark==0)&(supp==1),           "добавки без маркировки")

def tune_thr(y_true, proba):
    best_t, best_f1 = 0.5, -1.0
    for t in np.arange(0.20, 0.80, 0.02):
        s = f1_score(y_true, (proba >= t).astype(int), average='macro')
        if s > best_f1:
            best_f1, best_t = s, t
    return best_t, best_f1


artifact = {}
final_scores = {}

for name, tr_m, va_m in [("БАД", train_bad, val_bad),
                         ("Легковоспламеняющиеся", train_flam, val_flam)]:
    Xtr, ytr = X_train_f[tr_m], y_train[tr_m]
    Xva, yva = X_val_f[va_m],  y_val[va_m]

    lr = LogisticRegression(max_iter=2000, C=0.1, class_weight='balanced', random_state=42)
    lr.fit(Xtr, ytr)

    gb = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=15,
                            min_child_samples=20, class_weight='balanced',
                            random_state=42, verbose=-1)
    gb.fit(Xtr, ytr)

    p_lr, p_gb = lr.predict_proba(Xva)[:, 1], gb.predict_proba(Xva)[:, 1]
    p_ens = 0.5 * (p_lr + p_gb)

    print(f"\n=== {name} ===")
    cand = {}
    for tag, p in [("lr", p_lr), ("lgb", p_gb), ("ens", p_ens)]:
        t, f1 = tune_thr(yva, p)
        cand[tag] = (t, f1, p)
        print(f"{tag:4s} thr={t:.2f}  macroF1={f1:.4f}")

    best = max(cand, key=lambda k: cand[k][1])
    t_best, f1_best, p_best = cand[best]
    print(f">> выбран вариант: {best} (macroF1={f1_best:.4f})")

    artifact[name] = {"lr": lr, "lgb": gb, "thr": float(t_best), "variant": best}
    final_scores[name] = f1_best
    print(classification_report(yva, (p_best >= t_best).astype(int)))

print("\n>>> МЕТРИКА СОРЕВНОВАНИЯ (средний macro F1):", round(np.mean(list(final_scores.values())), 4))
joblib.dump(artifact, "/content/drive/MyDrive/Colab_Notebooks/classifiers_v4.pkl")
print("Сохранено: classifiers_v4.pkl")

Размерность признаков: 2068
--- Чистота правил (БАД, train) ---
спортпит (ждём ~0)                 n= 1932  доля label=1: 0.624
чистая маркировка (ждём ~1)        n= 3598  доля label=1: 0.878
добавки без маркировки (ждём ~0)   n=  397  доля label=1: 0.156


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



=== БАД ===
lr   thr=0.34  macroF1=0.8902
lgb  thr=0.44  macroF1=0.9069
ens  thr=0.44  macroF1=0.9051
>> выбран вариант: lgb (macroF1=0.9069)
              precision    recall  f1-score   support

           0       0.85      0.87      0.86       364
           1       0.96      0.95      0.95      1111

    accuracy                           0.93      1475
   macro avg       0.90      0.91      0.91      1475
weighted avg       0.93      0.93      0.93      1475



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



=== Легковоспламеняющиеся ===
lr   thr=0.66  macroF1=0.9114
lgb  thr=0.20  macroF1=0.9080
ens  thr=0.34  macroF1=0.9114
>> выбран вариант: lr (macroF1=0.9114)
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1078
           1       0.85      0.81      0.83        42

    accuracy                           0.99      1120
   macro avg       0.92      0.90      0.91      1120
weighted avg       0.99      0.99      0.99      1120


>>> МЕТРИКА СОРЕВНОВАНИЯ (средний macro F1): 0.9091
Сохранено: classifiers_v4.pkl
